# <center>**Enhancing LEM-X Imaging with the IROS Reconstruction Pipeline**<center>

## <center>**Sky Reconstruction Efficiency**<center>

In [1]:
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd

from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.io import simulation_files
from bloodmoon.types import CoordEquatorial
import darksun as ds
from darksun.data import Log, DataLoader, CatalogueLoader

from IROSrec.handle import config_dirpaths
import imgmaker as mgm
from imgmaker.fns import CameraUnitMap

In [2]:
# MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"
MASK_FITS: str = "mask_NTHT_20250725.fits"

# SKYFIELD: str = "IROSDummy"
SKYFIELD: str = "GalacticCentre"
# DATA_FITS: str = "baseline_2-50keV_1ks"
DATA_FITS: str = "galctr_rxte-sax_mask_050_1040x17_2-50keV_1ks_mask25"

RUN_ID: str = 'GC_IROS_doubleCam_rec_detected_2-6keV_UPX5_mask25'

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "detected"

E_min: float = 2.0  # [keV]
E_max: float = 6.0  # [keV]
coords2exclude: list[CoordEquatorial] | None = None

UPS_X, UPS_Y = 5, 1
hide_bulk_els_y: float = 1.5   # [mm]

In [3]:
MASK_PATH, SIMUL_DATA_PATH, SAVE_PATH = config_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
    runID=RUN_ID,
)
OUT_RESULTS_PATH = mgm.config_savedata_to()

wfm: CodedMaskCamera = codedmask(MASK_PATH, UPS_X, UPS_Y, hide_bulk_els_y=hide_bulk_els_y)

filepaths: dict[str, dict[str, Path]] = simulation_files(SIMUL_DATA_PATH)
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET], E_min=E_min, E_max=E_max, coords=coords2exclude)
catA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET], E_min=E_min, E_max=E_max, coords=coords2exclude)
catB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

logA, logB = ds.load_database(f"{SAVE_PATH}/IROS_sources_db.fits")

# Loading data...
# Loading completed!


### <center>**Benchmark Tables**<center>

In [4]:
import re

def adjust_Tabfrmt(txt: str) -> str:
    # insert \hline instead of rules (journal guidelines)
    for rule in ('toprule', 'midrule', 'bottomrule'):
        txt = txt.replace(rule, 'hline')
    # shift caption and label at the end (journal guidelines)
    pattern = r"(\\begin\{table\}.*?)(\\caption\{.*?\})\s*(\\label\{.*?\})\s*(\\begin\{tabular\}.*?\\end\{tabular\})"
    replacement = r"\1\4\n\2\n\3"
    txt = re.sub(pattern, replacement, txt, flags=re.DOTALL)
    # convert to onecolumn
    txt = txt.replace('table', 'table*')
    return txt

def sort_by(df: pd.DataFrame, key: str, **kwargs: Any) -> pd.DataFrame:
    """Sort DataFrame wrt input column key."""
    return df.sort_values(by=[key], ascending=False, ignore_index=True, **kwargs)

In [5]:
from numpy.typing import NDArray

def comp_src_mstd(log: Log, varmap: NDArray, boxsize: tuple[int, int]) -> NDArray:
    """Computes the RMSE for each IROS source from given varmap in specified array box."""
    mstds: list[float] = []
    boxsize_ = (max(boxsize[0], 1), max(boxsize[1], 1))
    for y, x in zip(log.log['y'], log.log['x']):
        srows, scols = (
            slice(y - boxsize_[0], y + boxsize_[0] + 1),
            slice(x - boxsize_[1], x + boxsize_[1] + 1),
        )
        mstd = np.sqrt(np.mean(varmap[srows, scols]))
        mstds.append(mstd)
    return np.array(mstds)


def gather_cam_data(
    log: Log,
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
    varmap: NDArray,
) -> pd.DataFrame:
    """
    Gathers single camera data from IROS reconstruction database.
    """
    ids = np.array([src.upper() for src in log.log['ID']])
    theta_res_x, theta_res_y = mgm.get_angularcoords_residues(log, catalogue, sdl, camera)
    cts = np.array(log.log['fluence']).round(decimals=0)
    true_cts = mgm.extract_catalogue_fluences(log, catalogue, sdl, camera)
    src_mstd = comp_src_mstd(log, varmap, boxsize=tuple(int(np.ceil(a // 2)) for a in ds.psf_extension(camera)[::-1]))
    dmap = {
        'Source': ids,
        'DthetaX': theta_res_x,
        'DthetaY': theta_res_y,
        'True_cts': true_cts,
        'IROS_cts': cts,
        'Dcts': (cts - true_cts) / src_mstd,
        'SNR': np.array(log.log['snr']),
    }
    return pd.DataFrame(dmap)


def get_unit_tab(data_camA: pd.DataFrame, data_camB: pd.DataFrame) -> pd.DataFrame:
    """Generates a Dataframe with output data from analysed LEM-X Unit."""
    # NOTE: data relative to not associated sources is DROPPED
    df = pd.merge(data_camA.dropna(), data_camB.dropna(), on='Source', how='outer', suffixes=('_A', '_B'))
    snrA, snrB = map(lambda col: df[col] ** 2, ('SNR_A', 'SNR_B'))
    df['SNR'] = np.sqrt(snrA.add(snrB, fill_value=0.0))
    df = df.drop(columns=['SNR_A', 'SNR_B'])
    return sort_by(df, 'SNR')

In [6]:
from bloodmoon.mask import count, variance

def get_varmap(camera: CodedMaskCamera, sdl: DataLoader) -> NDArray:
    detector = count(camera, sdl.DLdata)[0]
    varmap = variance(camera, detector)
    return varmap


varmapA, varmapB = map(lambda sdl: get_varmap(wfm, sdl), (sdlA, sdlB))

UserInfo: using bulk mask of [0.0 x 1.5] mm.


In [7]:
ds.pixels_angular_resolution(wfm)
cu_map = mgm.get_srcmap_for_unit(logA.log['ID'], logB.log['ID'])

# Table - CAMERA A
data_camA = gather_cam_data(logA, catA, sdlA, wfm, varmapA)

# Table - CAMERA B
data_camB = gather_cam_data(logB, catB, sdlB, wfm, varmapB)


Pixel angular resolution at upscaling (x, y): (5, 1)
  - fine direction: 0.8465 arcmin
  - coarse direction: 8.4653 arcmin



Analysing LEMX-CAM1BS3: 100%|██████████| 25/25 [00:01<00:00, 14.98it/s]  


In [8]:
unit_data = get_unit_tab(data_camA, data_camB)

KWS = {
    'label': 'Table1',
    'caption': 'Testing $`to\\_latex`$ fn.',
    'float_format': "%.4f",
    'column_format': 'l' + 'c' * (len(unit_data.columns) - 2) + 'r',
}
tab = mgm.df2TeXtab(
    df=sort_by(unit_data, 'SNR'),
    adjust_tabfrmt=adjust_Tabfrmt,
    # save_to=f'{OUT_RESULTS_PATH}/../texTable_Unit_results_{DATASET}_{E_min}-{E_max}keV_mask25.tex',
    overwrite=True,
    **KWS,
)

In [9]:
unit_data

,Source,DthetaX_A,DthetaY_A,True_cts_A,IROS_cts_A,Dcts_A,DthetaX_B,DthetaY_B,True_cts_B,IROS_cts_B,Dcts_B,SNR
0,SCOX1,0.031248,-0.294178,782305.0,779714.0,-2.472787,0.059871,-0.670794,696826.0,692129.0,-4.728606,920.350438
1,GX5-1,0.061258,-1.083480,87287.0,85969.0,-1.156590,0.188376,0.491512,88438.0,91239.0,2.519265,109.777429
2,GX349+2,0.089309,-0.405427,57654.0,57328.0,-0.285948,-0.007789,7.600004,57499.0,55189.0,-2.077125,62.940274
3,GX9+1,-0.056964,2.230578,45955.0,48693.0,2.401528,-0.017342,1.407160,46741.0,47078.0,0.303087,52.362987
4,GX17+2,0.159137,1.411388,44734.0,45168.0,0.384324,0.740859,1.352233,47708.0,40210.0,-6.754158,51.690429
5,GX13+1,0.103842,10.346890,27029.0,26805.0,-0.196475,0.338008,6.209383,26812.0,27016.0,0.183430,33.431827
6,GX3+1,0.177257,-6.702535,26883.0,28383.0,1.315977,0.039980,-5.562135,28060.0,27130.0,-0.836341,32.708583
7,GX340+0,-0.043781,4.979090,24080.0,23338.0,-0.686225,-0.088500,4.144554,25446.0,24499.0,-0.878726,31.535106
8,X1820-303,-0.269827,0.376444,20676.0,22288.0,1.413996,0.065200,-8.770179,20321.0,20723.0,0.361478,24.755856
9,GX9+9,-0.206174,-2.581237,18451.0,20187.0,1.522645,0.213969,-0.124272,18791.0,14520.0,-3.840635,22.307635


In [10]:
unit_data.dropna()

,Source,DthetaX_A,DthetaY_A,True_cts_A,IROS_cts_A,Dcts_A,DthetaX_B,DthetaY_B,True_cts_B,IROS_cts_B,Dcts_B,SNR
0,SCOX1,0.031248,-0.294178,782305.0,779714.0,-2.472787,0.059871,-0.670794,696826.0,692129.0,-4.728606,920.350438
1,GX5-1,0.061258,-1.083480,87287.0,85969.0,-1.156590,0.188376,0.491512,88438.0,91239.0,2.519265,109.777429
2,GX349+2,0.089309,-0.405427,57654.0,57328.0,-0.285948,-0.007789,7.600004,57499.0,55189.0,-2.077125,62.940274
3,GX9+1,-0.056964,2.230578,45955.0,48693.0,2.401528,-0.017342,1.407160,46741.0,47078.0,0.303087,52.362987
4,GX17+2,0.159137,1.411388,44734.0,45168.0,0.384324,0.740859,1.352233,47708.0,40210.0,-6.754158,51.690429
5,GX13+1,0.103842,10.346890,27029.0,26805.0,-0.196475,0.338008,6.209383,26812.0,27016.0,0.183430,33.431827
6,GX3+1,0.177257,-6.702535,26883.0,28383.0,1.315977,0.039980,-5.562135,28060.0,27130.0,-0.836341,32.708583
7,GX340+0,-0.043781,4.979090,24080.0,23338.0,-0.686225,-0.088500,4.144554,25446.0,24499.0,-0.878726,31.535106
8,X1820-303,-0.269827,0.376444,20676.0,22288.0,1.413996,0.065200,-8.770179,20321.0,20723.0,0.361478,24.755856
9,GX9+9,-0.206174,-2.581237,18451.0,20187.0,1.522645,0.213969,-0.124272,18791.0,14520.0,-3.840635,22.307635
